# CHW Workforce Operations Dashboard — Phase 3 Analysis

**All data are synthetic** (not NVCHWA administrative records). Snapshot: **2026-11-16**.

Run this notebook from the `analysis/` directory. Uses only Python standard library. The full phase-3 calculator is in `build_chw_phase3.py` under `analysis/`.

In [ ]:
from pathlib import Path
import csv, json, statistics
ROOT = Path.cwd().parent if Path.cwd().name == "analysis" else Path.cwd()
DATA = ROOT / "data"
ANALYSIS = ROOT / "analysis"
def read_csv(path):
    with path.open(newline="", encoding="utf-8-sig") as f:
        return list(csv.DictReader(f))
learners = read_csv(DATA / "learners.csv")
cohorts = read_csv(DATA / "cohorts.csv")
enrollments = read_csv(DATA / "enrollments.csv")
certification = read_csv(DATA / "certification_followup.csv")
employment = read_csv(DATA / "employment_followup.csv")
print({"learners":len(learners),"cohorts":len(cohorts),"enrollments":len(enrollments),"certification":len(certification),"employment":len(employment)})


## 1. Core KPIs

Completion and withdrawal denominators include only cohorts whose planned end date is on or before the snapshot. Enrollment and progress include all registrations, including still-active cohorts.

In [ ]:
summary = read_csv(ANALYSIS / "kpi_summary.csv")
for r in summary:
    print(f"{r['kpi_id']}: {r['kpi_name']} = {r['value'] or 'see breakdown'} {r['unit']}")


## 2. Calculate a core metric independently

This cell calculates K05 directly from enrollment records; it should agree with the published KPI table.

In [ ]:
eligible = [r for r in enrollments if r["completion_eligible"].lower() == "true"]
completed = [r for r in eligible if r["enrollment_status"] == "Completed"]
completion_pct = round(100 * len(completed) / len(eligible), 2)
print("Completion rate:", completion_pct, "%", "(", len(completed), "/", len(eligible), ")")
assert str(completion_pct) == next(r["value"] for r in summary if r["kpi_id"] == "K05")


## 3. Group comparisons

These precomputed tables use the exact same definitions as the KPI summary. Regional and language differences are descriptive patterns in simulated data.

In [ ]:
for filename in ("kpi_by_year.csv", "kpi_by_region.csv", "kpi_by_language.csv"):
    print("\n", filename)
    for r in read_csv(ANALYSIS / filename):
        label = r.get("year", r.get("region", r.get("training_language")))
        print(label, "enrollments:", r["enrollments"], "completion:", r["completion_rate_pct"], "%")


## 4. Follow-up outcomes and selection

Only 3- or 6-month eligible completers are in each coverage denominator. Unknown employment status is **not** treated as unemployment; observed CHW employment rates exclude Unknown. Certification uses the latest valid post-completion milestone after the most recent completion, and does not represent the share of all completers certified.

In [ ]:
for r in read_csv(ANALYSIS / "kpi_followup_windows.csv"):
    print(f"{r['followup_month']}-month: {r['observed_unique_completers']}/{r['eligible_unique_completers']} observed; coverage {r['followup_coverage_pct']}%; known employment {r['known_status_observations']}; observed CHW employment {r['observed_chw_employment_rate_pct']}%")


## 5. Check validation and build dashboard exports

`build_chw_phase3.py` is a fully reproducible standard-library calculator. Run it when updating Phase 2 CSVs, then rerun this notebook.

In [ ]:
report = json.loads((ROOT / "documentation" / "PHASE_3_VALIDATION_REPORT.json").read_text())
assert report["all_passed"]
print("Validation checks:", len(report["checks"]), "passed")
print("Dashboard enrollment rows:", len(read_csv(ROOT / "dashboard" / "looker_enrollment_detail.csv")))


## 6. Interpretation and limitations

- All figures are **fabricated demonstration values**, never Nevada CHW benchmark statistics.
- A learner with repeat enrollment counts once for K02, but multiple times for K01.
- Follow-up data are limited to those with qualifying training completion and observed records.
- Differences between groups and employment outcomes are **not causal evidence**.
- Cohort fill rate can exceed 100% by design; do not cap at 100%.
- Quarterly completion trends group enrollments by their cohort planned end quarter, not by individual completion date.
- CHW retention is persistence of CHW employment status from 3 to 6 months, not same-employer tenure.
